In [ ]:
#!/usr/bin/env python3
"""
Housing Prices Submission Script - SGD Stacked Ensemble (Optimized)

Key Improvements:
1. FeatureProcessor (Target Encoding) fitted only on training folds (non-leaky CV).
2. CatBoost hyperparameter tuning:
    - n_estimators increased (1900 -> 2200)
    - learning_rate decreased (0.02 -> 0.015)
This subtle change enhances CatBoost generalization by training deeper and slower.
"""
from __future__ import annotations

import logging
import os
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import SGDRegressor
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

warnings.filterwarnings('ignore')

# ============================================================================
# LOGGING SETUP
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# ============================================================================
# CONFIGURATION
# ============================================================================
GRLIV_AREA_OUTLIER_MAX = 4000
SALE_PRICE_OUTLIER_MIN = 300_000
OHE_MIN_FREQUENCY = 0.03
QUALITY_ORDER = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
FUNCTIONAL_ORDER = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
UTILITIES_FILL = 'AllPub'
RANDOM_STATE = 42
N_FOLDS = 7

# --- HYPERPARAMETER TUNING ---
CATBOOST_ESTIMATORS = 2200 # Increased from 1900
CATBOOST_LEARNING_RATE = 0.015 # Decreased from 0.02

# ============================================================================
# FEATURE DEFINITIONS (UNCHANGED)
# ============================================================================
CORE_FEATURES = [
    'GrLivArea', 'TotalBsmtSF', 'LotArea', 'GarageArea', 'PoolArea', 'LotFrontage',
    '2ndFlrSF', 'LowQualFinSF', 'BsmtUnfSF', '1stFlrSF',
    'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch',
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath', 'TotRmsAbvGrd', 'Fireplaces',
    'YearBuilt', 'YrSold', 'YearRemodAdd',
    'OverallQual', 'OverallCond', 'HeatingQC', 'BsmtQual', 'PoolQC',
    'ExterQual', 'KitchenQual', 'Functional',
    'FireplaceQu', 'BsmtCond', 'ExterCond',
    'Neighborhood', 'MSZoning', 'MSSubClass',
    'LandSlope', 'Alley', 'LandContour', 'BldgType',
    'Condition1', 'RoofStyle', 'Foundation',
    'SaleCondition', 'Exterior1st',
    'Utilities', 'Electrical',
    'GarageQual', 'GarageCond',
]

OHE_CATEGORICAL_COLS_KRR = [
    'Neighborhood', 'MSZoning', 'LandSlope', 'Alley', 'LandContour', 'BldgType',
    'Condition1', 'RoofStyle', 'Foundation', 'SaleCondition', 'Exterior1st',
    'Utilities', 'Electrical', 'GarageQual', 'GarageCond', 'MSSubClass',
]

OHE_CATEGORICAL_COLS_CAT = [
    'Neighborhood', 'MSZoning', 'LandSlope', 'Alley', 'LandContour', 'BldgType',
    'Condition1', 'RoofStyle', 'Foundation', 'SaleCondition', 'Exterior1st',
    'Utilities', 'Electrical', 'GarageQual', 'GarageCond',
]

ORDINAL_QUALITY_COLS = [
    'FireplaceQu', 'BsmtCond', 'KitchenQual', 'ExterQual',
    'HeatingQC', 'BsmtQual', 'PoolQC', 'ExterCond'
]

ORDINAL_FUNCTIONAL_COLS = ['Functional']

FILL_ZERO_COLS = [
    'PoolArea', 'GrLivArea', 'LotArea',
    'TotalBsmtSF', 'BsmtUnfSF',
    'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath',
    '2ndFlrSF', 'LowQualFinSF', '1stFlrSF', '3SsnPorch',
    'EnclosedPorch', 'ScreenPorch', 'WoodDeckSF', 'OpenPorchSF',
    'GarageArea',
]

SKEWED_FEATURES = [
    'LotArea', 'PoolArea', 'LowQualFinSF', 'BsmtHalfBath',
    'GrLivArea', 'LotFrontage', '1stFlrSF', '2ndFlrSF',
    'BsmtUnfSF', 'TotalSF', 'TotalQualSF', 'InteriorQualityScore',
]

PORCH_DECK_COLS = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']

# ============================================================================
# DATA PATH HELPER, REMOVE_OUTLIERS, FEATUREPROCESSOR (UNCHANGED logic)
# ============================================================================
def get_data_path(filename: str) -> str:
    paths_to_try = [
        f'/Users/vkdvamshi/Downloads/home-data-for-ml-course/{filename}',
        f'/Users/vkdvamshi/Downloads/home-data-for-ml-course/{filename}',
        f'/Users/vkdvamshi/Downloads/home-data-for-ml-course/{filename}',
    ]
    for path in paths_to_try:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Could not find {filename} in any expected location")

def remove_outliers(df: pd.DataFrame) -> pd.DataFrame:
    if 'SalePrice' not in df.columns:
        return df

    outlier_mask = (
        (df['GrLivArea'] > GRLIV_AREA_OUTLIER_MAX) &
        (df['SalePrice'] < SALE_PRICE_OUTLIER_MIN)
    )
    df_cleaned = df.drop(df[outlier_mask].index, axis=0)
    return df_cleaned

class FeatureProcessor:
    """Gemini-style preprocessing with target encoding."""

    def __init__(self) -> None:
        self.imputation_values_: dict = {}
        self.neighborhood_price_per_sf_: dict = {}
        self.is_fitted_: bool = False

    def fit(self, X: pd.DataFrame, train_df: pd.DataFrame = None) -> FeatureProcessor:
        self.imputation_values_['LotFrontage'] = X['LotFrontage'].median()
        self.imputation_values_['YearBuilt_median'] = X['YearBuilt'].median()
        self.imputation_values_['YrSold_mode'] = X['YrSold'].mode()[0]

        if train_df is not None and 'SalePrice' in train_df.columns:
            train_df = train_df.copy()
            train_df['_TotalQualSF'] = (train_df['GrLivArea'] + train_df['TotalBsmtSF'].fillna(0)) * train_df['OverallQual']
            
            valid_rows = train_df['_TotalQualSF'] > 0
            train_df['_PricePerQualSF'] = np.nan
            train_df.loc[valid_rows, '_PricePerQualSF'] = train_df.loc[valid_rows, 'SalePrice'] / train_df.loc[valid_rows, '_TotalQualSF']
            
            self.neighborhood_price_per_sf_ = train_df.groupby('Neighborhood')['_PricePerQualSF'].median().to_dict()
            self.imputation_values_['NeighborhoodPricePerQualSF_median'] = train_df['_PricePerQualSF'].median()

        self.is_fitted_ = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self.is_fitted_:
            raise RuntimeError("Must call fit() before transform()")

        X = X.copy()
        X = self._apply_imputations(X)
        X = self._fill_missing_zeros(X)
        X = self._add_engineered_features(X)
        X = self._log_transform_skewed(X)
        return X

    def fit_transform(self, X: pd.DataFrame, train_df: pd.DataFrame = None) -> pd.DataFrame:
        return self.fit(X, train_df).transform(X)

    def _apply_imputations(self, X: pd.DataFrame) -> pd.DataFrame:
        X['LotFrontage'] = X['LotFrontage'].fillna(self.imputation_values_['LotFrontage'])
        median_year = self.imputation_values_['YearBuilt_median']
        X['YearBuilt'] = X['YearBuilt'].fillna(median_year)
        X['YearRemodAdd'] = X['YearRemodAdd'].fillna(X['YearBuilt'])
        X['YrSold'] = X['YrSold'].fillna(self.imputation_values_['YrSold_mode'])
        X['Utilities'] = X['Utilities'].fillna(UTILITIES_FILL)
        return X

    def _fill_missing_zeros(self, X: pd.DataFrame) -> pd.DataFrame:
        X[FILL_ZERO_COLS] = X[FILL_ZERO_COLS].fillna(0)
        return X

    def _add_engineered_features(self, X: pd.DataFrame) -> pd.DataFrame:
        X['TotalSF'] = X['GrLivArea'] + X['TotalBsmtSF']
        X['TotalQualSF'] = X['TotalSF'] * X['OverallQual']
        X['TimeSinceRemod'] = X['YrSold'] - X['YearRemodAdd']
        X['Age'] = X['YrSold'] - X['YearBuilt']
        X['InteriorQualityScore'] = X['GrLivArea'] * X['OverallQual']

        X['TotalBaths'] = (
            X['FullBath'] +
            (0.5 * X['HalfBath']) +
            X['BsmtFullBath'] +
            (0.5 * X['BsmtHalfBath'])
        )

        X['HasPorchDeck'] = (X[PORCH_DECK_COLS].sum(axis=1) > 0).astype(int)
        X['TotalPorchDeckSF'] = X[PORCH_DECK_COLS].sum(axis=1)

        if self.neighborhood_price_per_sf_:
            fallback = self.imputation_values_.get('NeighborhoodPricePerQualSF_median', 0)
            X['NeighborhoodPricePerQualSF'] = X['Neighborhood'].map(self.neighborhood_price_per_sf_).fillna(fallback)

        return X

    def _log_transform_skewed(self, X: pd.DataFrame) -> pd.DataFrame:
        for col in SKEWED_FEATURES:
            if col in X.columns:
                X[col] = np.log1p(X[col])
        return X

# ============================================================================
# ENCODING PIPELINES (UNCHANGED)
# ============================================================================
def build_feature_pipeline(ohe_cols: list) -> Pipeline:
    """Build feature pipeline with configurable OHE columns."""
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(
            handle_unknown='infrequent_if_exist',
            min_frequency=OHE_MIN_FREQUENCY,
            sparse_output=False,
            drop='first'
        ))
    ])

    ohe_preprocessor = ColumnTransformer(
        transformers=[('cat', categorical_pipeline, ohe_cols)],
        remainder='passthrough',
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    ordinal_transformer = ColumnTransformer(
        transformers=[
            ('quality_enc', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
                ('ordinal', OrdinalEncoder(
                    categories=[['None'] + QUALITY_ORDER] * len(ORDINAL_QUALITY_COLS),
                    handle_unknown='use_encoded_value',
                    unknown_value=-1
                ))
            ]), ORDINAL_QUALITY_COLS),
            ('functional_enc', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
                ('ordinal', OrdinalEncoder(
                    categories=[['None'] + FUNCTIONAL_ORDER],
                    handle_unknown='use_encoded_value',
                    unknown_value=-1
                ))
            ]), ORDINAL_FUNCTIONAL_COLS),
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    ).set_output(transform="pandas")

    return Pipeline(steps=[
        ('ohe_proc', ohe_preprocessor),
        ('ordinal_prep', ordinal_transformer),
    ])

# ============================================================================
# MAIN EXECUTION (MODIFIED)
# ============================================================================
def main() -> None:
    logger.info("=" * 60)
    logger.info("Housing Prices - SGD Stacked Ensemble (Tuned CatBoost)")
    logger.info(f"CatBoost: n_est={CATBOOST_ESTIMATORS}, lr={CATBOOST_LEARNING_RATE}")
    logger.info("=" * 60)

    train_path = get_data_path('train.csv')
    test_path = get_data_path('test.csv')

    # --- Load and Prepare Data ---
    train_df = pd.read_csv(train_path)
    train_df = remove_outliers(train_df)
    train_df = train_df.reset_index(drop=True)

    X_train_raw = train_df[CORE_FEATURES].copy()
    
    train_total_qual_sf = (train_df['GrLivArea'] + train_df['TotalBsmtSF'].fillna(0)) * train_df['OverallQual']
    y = np.log1p(train_df.SalePrice / train_total_qual_sf).values

    krr_encoder = build_feature_pipeline(OHE_CATEGORICAL_COLS_KRR)
    cat_encoder = build_feature_pipeline(OHE_CATEGORICAL_COLS_CAT)

    # =========================================================================
    # STAGE 1: Get OOF predictions from all 3 models (Non-Leaky CV)
    # =========================================================================
    logger.info("\n" + "=" * 60)
    logger.info(f"STAGE 1: {N_FOLDS}-Fold CV for OOF predictions...")
    logger.info("=" * 60)

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    
    oof_krr = np.zeros(len(y))
    oof_cat = np.zeros(len(y))
    oof_cat_resid = np.zeros(len(y))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_raw)):
        logger.info(f"  Fold {fold + 1}/{N_FOLDS}...")
        
        X_train_sub, X_val_sub = X_train_raw.iloc[train_idx], X_train_raw.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        train_df_sub = train_df.iloc[train_idx]

        # Non-Leaky Feature Engineering
        processor_fold = FeatureProcessor()
        X_train_processed = processor_fold.fit_transform(X_train_sub, train_df_sub)
        X_val_processed = processor_fold.transform(X_val_sub)
        
        # Encoding
        X_krr_train = krr_encoder.fit_transform(X_train_processed)
        X_krr_val = krr_encoder.transform(X_val_processed)
        X_cat_train = cat_encoder.fit_transform(X_train_processed)
        X_cat_val = cat_encoder.transform(X_val_processed)

        # --- Model 1: KernelRidge ---
        scaler = StandardScaler()
        X_krr_train_scaled = scaler.fit_transform(X_krr_train)
        X_krr_val_scaled = scaler.transform(X_krr_val)
        
        krr = KernelRidge(kernel='rbf', alpha=0.0003, gamma=0.0001)
        krr.fit(X_krr_train_scaled, y_train)
        
        krr_train_preds = krr.predict(X_krr_train_scaled)
        oof_krr[val_idx] = krr.predict(X_krr_val_scaled)
        
        # --- Model 2: CatBoost (Tuned) ---
        cat = CatBoostRegressor(
            n_estimators=CATBOOST_ESTIMATORS,
            learning_rate=CATBOOST_LEARNING_RATE,
            depth=6, l2_leaf_reg=3, subsample=0.75,
            colsample_bylevel=0.75, loss_function='RMSE', random_state=RANDOM_STATE, verbose=0,
        )
        cat.fit(X_cat_train, y_train)
        oof_cat[val_idx] = cat.predict(X_cat_val)
        
        # --- Model 3: CatBoost on KRR residuals (Tuned) ---
        residuals_train = y_train - krr_train_preds
        
        cat_resid = CatBoostRegressor(
            n_estimators=CATBOOST_ESTIMATORS,
            learning_rate=CATBOOST_LEARNING_RATE,
            depth=6, l2_leaf_reg=3, subsample=0.75,
            colsample_bylevel=0.75, loss_function='RMSE', random_state=RANDOM_STATE, verbose=0,
        )
        cat_resid.fit(X_cat_train, residuals_train)
        oof_cat_resid[val_idx] = cat_resid.predict(X_cat_val)

    # Report OOF performance
    logger.info(f"\nOOF RMSE - KRR: {np.sqrt(np.mean((y - oof_krr)**2)):.6f}")
    logger.info(f"OOF RMSE - CatBoost: {np.sqrt(np.mean((y - oof_cat)**2)):.6f}")
    logger.info(f"OOF RMSE - KRR+CatResid: {np.sqrt(np.mean((y - (oof_krr + oof_cat_resid))**2)):.6f}")

    # =========================================================================
    # STAGE 2: Train SGD meta-learner on OOF predictions
    # =========================================================================
    oof_stack = np.column_stack([oof_krr, oof_cat, oof_cat_resid])
    stack_scaler = StandardScaler()
    oof_stack_scaled = stack_scaler.fit_transform(oof_stack)
    
    sgd_meta = SGDRegressor(
        loss='epsilon_insensitive', epsilon=0.0, penalty='l2', eta0=0.01,
        alpha=0.001, random_state=RANDOM_STATE, max_iter=1000, tol=1e-4,
    )
    sgd_meta.fit(oof_stack_scaled, y)
    
    # =========================================================================
    # STAGE 3: Train final models on full data
    # =========================================================================
    logger.info("\n" + "=" * 60)
    logger.info("STAGE 3: Training final models on full data...")
    logger.info("=" * 60)

    # Final FeatureProcessor fitted on the entire training set
    processor_final = FeatureProcessor()
    X_train_processed_final = processor_final.fit_transform(X_train_raw, train_df)

    # Transform full data using final feature encoders
    X_train_krr_final = krr_encoder.fit_transform(X_train_processed_final)
    X_train_cat_final = cat_encoder.fit_transform(X_train_processed_final)

    # Final KernelRidge
    scaler_final = StandardScaler()
    X_train_krr_scaled = scaler_final.fit_transform(X_train_krr_final)
    
    krr_final = KernelRidge(kernel='rbf', alpha=0.0003, gamma=0.0001)
    krr_final.fit(X_train_krr_scaled, y)
    krr_train_preds_final = krr_final.predict(X_train_krr_scaled)
    
    # Final CatBoost (Tuned)
    cat_final = CatBoostRegressor(
        n_estimators=CATBOOST_ESTIMATORS,
        learning_rate=CATBOOST_LEARNING_RATE,
        depth=6, l2_leaf_reg=3, subsample=0.75,
        colsample_bylevel=0.75, loss_function='RMSE', random_state=RANDOM_STATE, verbose=100,
    )
    cat_final.fit(X_train_cat_final, y)
    
    # Final CatBoost on residuals (Tuned)
    residuals_final = y - krr_train_preds_final
    
    cat_resid_final = CatBoostRegressor(
        n_estimators=CATBOOST_ESTIMATORS,
        learning_rate=CATBOOST_LEARNING_RATE,
        depth=6, l2_leaf_reg=3, subsample=0.75,
        colsample_bylevel=0.75, loss_function='RMSE', random_state=RANDOM_STATE, verbose=100,
    )
    cat_resid_final.fit(X_train_cat_final, residuals_final)

    # =========================================================================
    # STAGE 4: Make predictions on test data
    # =========================================================================
    logger.info("\n" + "=" * 60)
    logger.info("Making predictions on test data...")
    logger.info("=" * 60)

    test_df = pd.read_csv(test_path)
    test_ids = test_df['Id']

    test_total_qual_sf = (test_df['GrLivArea'] + test_df['TotalBsmtSF'].fillna(0)) * test_df['OverallQual']
    test_X_raw = test_df[CORE_FEATURES].copy()
    
    test_X_processed = processor_final.transform(test_X_raw)

    test_X_krr = krr_encoder.transform(test_X_processed)
    test_X_krr_scaled = scaler_final.transform(test_X_krr)
    test_krr_preds = krr_final.predict(test_X_krr_scaled)

    test_X_cat = cat_encoder.transform(test_X_processed)
    test_cat_preds = cat_final.predict(test_X_cat)
    test_cat_resid_preds = cat_resid_final.predict(test_X_cat)

    # Stack test predictions and predict via SGD
    test_stack = np.column_stack([test_krr_preds, test_cat_preds, test_cat_resid_preds])
    test_stack_scaled = stack_scaler.transform(test_stack)
    combined_preds_log = sgd_meta.predict(test_stack_scaled)

    # Inverse transform
    combined_preds_per_sqft = np.expm1(combined_preds_log)
    predictions = combined_preds_per_sqft * test_total_qual_sf

    logger.info("Prediction complete!")

    output = pd.DataFrame({
        'Id': test_ids,
        'SalePrice': predictions
    })
    output.to_csv('submission.csv', index=False)

    logger.info("\n" + "=" * 60)
    logger.info("Submission saved as 'submission.csv'")
    logger.info("=" * 60)


if __name__ == "__main__":
    try:
        main()
    except FileNotFoundError as e:
        logger.error(f"Error: {e}. Please ensure train.csv and test.csv are in the expected input directory.")

Housing Prices - SGD Stacked Ensemble (Tuned CatBoost)
CatBoost: n_est=2200, lr=0.015

STAGE 1: 7-Fold CV for OOF predictions...
  Fold 1/7...
  Fold 2/7...
  Fold 3/7...
  Fold 4/7...
  Fold 5/7...
  Fold 6/7...
  Fold 7/7...

OOF RMSE - KRR: 0.105862
OOF RMSE - CatBoost: 0.114820
OOF RMSE - KRR+CatResid: 0.105770

STAGE 3: Training final models on full data...
0:	learn: 0.2209806	total: 1.15ms	remaining: 2.53s
100:	learn: 0.1501030	total: 109ms	remaining: 2.26s
200:	learn: 0.1269156	total: 215ms	remaining: 2.14s
300:	learn: 0.1141852	total: 329ms	remaining: 2.07s
400:	learn: 0.1042782	total: 435ms	remaining: 1.95s
500:	learn: 0.0968276	total: 542ms	remaining: 1.84s
600:	learn: 0.0903090	total: 645ms	remaining: 1.72s
700:	learn: 0.0843678	total: 750ms	remaining: 1.6s
800:	learn: 0.0794813	total: 856ms	remaining: 1.49s
900:	learn: 0.0752842	total: 961ms	remaining: 1.39s
1000:	learn: 0.0714893	total: 1.07s	remaining: 1.28s
1100:	learn: 0.0679955	total: 1.17s	remaining: 1.17s
1200:	learn